Steps used:

 imports → stage → load → profile → clean → schema → dimensions → facts → accumulating snapshot → periodic snapshot → reconcile

In [ ]:
import os
import math
import shutil
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import (create_engine, MetaData, Table, Column, Integer,
                        SmallInteger, String, Numeric, Float, Date, DateTime,
                        Text, ForeignKey, PrimaryKeyConstraint, text)

RAW_DIR = "."            # source CSVs live in the notebook's folder
STAGING = "staging"      # raw layer (unmodified copies)
DATABASE_URL = os.getenv("DATABASE_URL", "postgresql+psycopg2://postgres:<password>@localhost:5432/olist_dw")

FILES = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

In [4]:
os.makedirs(STAGING, exist_ok=True)
for fname in FILES.values():
    shutil.copyfile(os.path.join(RAW_DIR, fname), os.path.join(STAGING, fname))
print(f"Staged {len(FILES)} files unmodified into '{STAGING}/'.")

Staged 9 files unmodified into 'staging/'.


In [5]:
orders      = pd.read_csv(f"{STAGING}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{STAGING}/olist_order_items_dataset.csv")
payments    = pd.read_csv(f"{STAGING}/olist_order_payments_dataset.csv")
reviews     = pd.read_csv(f"{STAGING}/olist_order_reviews_dataset.csv")
customers   = pd.read_csv(f"{STAGING}/olist_customers_dataset.csv",
                          dtype={"customer_zip_code_prefix": str})
products    = pd.read_csv(f"{STAGING}/olist_products_dataset.csv")
sellers     = pd.read_csv(f"{STAGING}/olist_sellers_dataset.csv",
                          dtype={"seller_zip_code_prefix": str})
geolocation = pd.read_csv(f"{STAGING}/olist_geolocation_dataset.csv",
                          dtype={"geolocation_zip_code_prefix": str})
translation = pd.read_csv(f"{STAGING}/product_category_name_translation.csv")
print("Loaded staged data into DataFrames.")

Loaded staged data into DataFrames.


In [6]:
EXPECTED_ROWS = {"orders": 99441, "order_items": 112650, "payments": 103886,
                 "reviews": 99224, "customers": 99441, "products": 32951,
                 "sellers": 3095, "geolocation": 1000163, "translation": 71}

print("A. ROW COUNTS (actual vs table_info.md)")
for key, df in [("orders", orders), ("order_items", order_items), ("payments", payments),
                ("reviews", reviews), ("customers", customers), ("products", products),
                ("sellers", sellers), ("geolocation", geolocation), ("translation", translation)]:
    ok = "OK" if len(df) == EXPECTED_ROWS[key] else "!! MISMATCH"
    print(f"   {key:12s} actual={len(df):>9,}  expected={EXPECTED_ROWS[key]:>9,}  {ok}")

print("\nB. DTYPES")
for key, df in [("orders", orders), ("order_items", order_items), ("payments", payments),
                ("reviews", reviews), ("customers", customers), ("products", products),
                ("sellers", sellers), ("geolocation", geolocation), ("translation", translation)]:
    print(f"   [{key}] " + " | ".join(f"{c}:{df[c].dtype}" for c in df.columns))

print("\nC. NULLS (only columns with nulls)")
for key, df in [("orders", orders), ("reviews", reviews), ("products", products)]:
    n = df.isna().sum(); n = n[n > 0]
    print(f"   [{key}]")
    print("    " + (n.to_string().replace("\n", "\n    ") if len(n) else "(none)"))

print("\nD. DUPLICATE review_id (key data-quality finding)")
print(f"   duplicate review_id rows = {int(reviews.duplicated(subset=['review_id']).sum()):,}")

A. ROW COUNTS (actual vs table_info.md)
   orders       actual=   99,441  expected=   99,441  OK
   order_items  actual=  112,650  expected=  112,650  OK
   payments     actual=  103,886  expected=  103,886  OK
   reviews      actual=   99,224  expected=   99,224  OK
   customers    actual=   99,441  expected=   99,441  OK
   products     actual=   32,951  expected=   32,951  OK
   sellers      actual=    3,095  expected=    3,095  OK
   geolocation  actual=1,000,163  expected=1,000,163  OK
   translation  actual=       71  expected=       71  OK

B. DTYPES
   [orders] order_id:str | customer_id:str | order_status:str | order_purchase_timestamp:str | order_approved_at:str | order_delivered_carrier_date:str | order_delivered_customer_date:str | order_estimated_delivery_date:str
   [order_items] order_id:str | order_item_id:int64 | product_id:str | seller_id:str | shipping_limit_date:str | price:float64 | freight_value:float64
   [payments] order_id:str | payment_sequential:int64 | payme

In [7]:
def orphans(child, ccol, parent, pcol):
    pset = set(parent[pcol].dropna())
    return int((~child[ccol].dropna().isin(pset)).sum())

print("E. CONSISTENCY (value rules)")
checks = [
    ("orders.order_status valid", orders["order_status"].isin(
        ["created", "approved", "invoiced", "processing", "shipped",
         "delivered", "unavailable", "canceled"]).all()),
    ("payments.payment_type valid", payments["payment_type"].isin(
        ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]).all()),
    ("reviews.review_score in 1..5", reviews["review_score"].between(1, 5).all()),
    ("order_items.price >= 0", pd.to_numeric(order_items["price"]).ge(0).all()),
    ("order_items.freight_value >= 0", pd.to_numeric(order_items["freight_value"]).ge(0).all()),
    ("payments.payment_value >= 0", pd.to_numeric(payments["payment_value"]).ge(0).all()),
    ("payments.payment_installments >= 0", pd.to_numeric(payments["payment_installments"]).ge(0).all()),
]
for label, ok in checks:
    print(f"   {'PASS' if ok else 'FAIL'}  {label}")

print("\nF. REFERENTIAL INTEGRITY (orphan counts)")
rel = [
    (orders, "customer_id", customers, "customer_id", "orders.customer_id -> customers"),
    (order_items, "order_id", orders, "order_id", "order_items.order_id -> orders"),
    (order_items, "product_id", products, "product_id", "order_items.product_id -> products"),
    (order_items, "seller_id", sellers, "seller_id", "order_items.seller_id -> sellers"),
    (payments, "order_id", orders, "order_id", "payments.order_id -> orders"),
    (reviews, "order_id", orders, "order_id", "reviews.order_id -> orders"),
]
for child, ccol, parent, pcol, lbl in rel:
    print(f"   orphans={orphans(child, ccol, parent, pcol):>6,}  {lbl}")

E. CONSISTENCY (value rules)
   PASS  orders.order_status valid
   PASS  payments.payment_type valid
   PASS  reviews.review_score in 1..5
   PASS  order_items.price >= 0
   PASS  order_items.freight_value >= 0
   PASS  payments.payment_value >= 0
   PASS  payments.payment_installments >= 0

F. REFERENTIAL INTEGRITY (orphan counts)
   orphans=     0  orders.customer_id -> customers
   orphans=     0  order_items.order_id -> orders
   orphans=     0  order_items.product_id -> products
   orphans=     0  order_items.seller_id -> sellers
   orphans=     0  payments.order_id -> orders
   orphans=     0  reviews.order_id -> orders


In [8]:
def orphans(child, ccol, parent, pcol):
    pset = set(parent[pcol].dropna())
    return int((~child[ccol].dropna().isin(pset)).sum())

print("E. CONSISTENCY (value rules)")
checks = [
    ("orders.order_status valid", orders["order_status"].isin(
        ["created", "approved", "invoiced", "processing", "shipped",
         "delivered", "unavailable", "canceled"]).all()),
    ("payments.payment_type valid", payments["payment_type"].isin(
        ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]).all()),
    ("reviews.review_score in 1..5", reviews["review_score"].between(1, 5).all()),
    ("order_items.price >= 0", pd.to_numeric(order_items["price"]).ge(0).all()),
    ("order_items.freight_value >= 0", pd.to_numeric(order_items["freight_value"]).ge(0).all()),
    ("payments.payment_value >= 0", pd.to_numeric(payments["payment_value"]).ge(0).all()),
    ("payments.payment_installments >= 0", pd.to_numeric(payments["payment_installments"]).ge(0).all()),
]
for label, ok in checks:
    print(f"   {'PASS' if ok else 'FAIL'}  {label}")

print("\nF. REFERENTIAL INTEGRITY (orphan counts)")
rel = [
    (orders, "customer_id", customers, "customer_id", "orders.customer_id -> customers"),
    (order_items, "order_id", orders, "order_id", "order_items.order_id -> orders"),
    (order_items, "product_id", products, "product_id", "order_items.product_id -> products"),
    (order_items, "seller_id", sellers, "seller_id", "order_items.seller_id -> sellers"),
    (payments, "order_id", orders, "order_id", "payments.order_id -> orders"),
    (reviews, "order_id", orders, "order_id", "reviews.order_id -> orders"),
]
for child, ccol, parent, pcol, lbl in rel:
    print(f"   orphans={orphans(child, ccol, parent, pcol):>6,}  {lbl}")

E. CONSISTENCY (value rules)
   PASS  orders.order_status valid
   PASS  payments.payment_type valid
   PASS  reviews.review_score in 1..5
   PASS  order_items.price >= 0
   PASS  order_items.freight_value >= 0
   PASS  payments.payment_value >= 0
   PASS  payments.payment_installments >= 0

F. REFERENTIAL INTEGRITY (orphan counts)
   orphans=     0  orders.customer_id -> customers
   orphans=     0  order_items.order_id -> orders
   orphans=     0  order_items.product_id -> products
   orphans=     0  order_items.seller_id -> sellers
   orphans=     0  payments.order_id -> orders
   orphans=     0  reviews.order_id -> orders


In [9]:
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")
print("Timestamps parsed (null milestones remain NaT).")

Timestamps parsed (null milestones remain NaT).


In [10]:
for df, col in [(customers, "customer_zip_code_prefix"),
                (sellers, "seller_zip_code_prefix"),
                (geolocation, "geolocation_zip_code_prefix")]:
    df[col] = df[col].astype(str).str.strip().str.zfill(5)
print("ZIP prefixes zero-padded to 5 chars.")

ZIP prefixes zero-padded to 5 chars.


In [11]:
products = products.rename(columns={
    "product_name_lenght":        "product_name_length",
    "product_description_lenght": "product_description_length",
})
translation["product_category_name_english"] = translation["product_category_name_english"].replace({
    "costruction_tools_garden": "construction_tools_garden",
    "costruction_tools_tools":  "construction_tools_tools",
    "home_confort":             "home_comfort",
    "fashio_female_clothing":   "fashion_female_clothing",
    "arts_and_craftmanship":    "arts_and_crafts",
})
print("Renamed typo columns; fixed translation typos.")

Renamed typo columns; fixed translation typos.


In [12]:
for c in ["review_comment_title", "review_comment_message"]:
    reviews[c] = reviews[c].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

products["product_category_name"] = products["product_category_name"].fillna("unknown")
for c in ["product_name_length", "product_description_length", "product_photos_qty"]:
    products[c] = products[c].fillna(0).astype(int)
for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[c] = products[c].fillna(products[c].median()).round().astype(int)
if "unknown" not in set(translation["product_category_name"]):
    translation = pd.concat([translation, pd.DataFrame(
        [{"product_category_name": "unknown",
          "product_category_name_english": "unknown"}])], ignore_index=True)
print("Review text normalized; product gaps filled.")

Review text normalized; product gaps filled.


In [14]:
def drop_orphans(child, ccol, parent, pcol, label):
    pset = set(parent[pcol].dropna())
    mask = child[ccol].isin(pset)
    n = int((~mask).sum())
    print(f"   {'drop' if n else 'ok  '} {n:>6,} orphans  {label}")
    return child[mask].reset_index(drop=True)

orders      = drop_orphans(orders, "customer_id", customers, "customer_id", "orders.customer_id -> customers")
order_items = drop_orphans(order_items, "order_id", orders, "order_id", "order_items.order_id -> orders")
order_items = drop_orphans(order_items, "product_id", products, "product_id", "order_items.product_id -> products")
order_items = drop_orphans(order_items, "seller_id", sellers, "seller_id", "order_items.seller_id -> sellers")
payments    = drop_orphans(payments, "order_id", orders, "order_id", "payments.order_id -> orders")
reviews     = drop_orphans(reviews, "order_id", orders, "order_id", "reviews.order_id -> orders")

   ok        0 orphans  orders.customer_id -> customers
   ok        0 orphans  order_items.order_id -> orders
   ok        0 orphans  order_items.product_id -> products
   ok        0 orphans  order_items.seller_id -> sellers
   ok        0 orphans  payments.order_id -> orders
   ok        0 orphans  reviews.order_id -> orders


In [15]:
n_geo = len(geolocation)
geolocation = geolocation.drop_duplicates(
    subset=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    keep="first").reset_index(drop=True)
print(f"geolocation: {n_geo:,} -> {len(geolocation):,} ({n_geo - len(geolocation):,} dup coords)")

orders      = orders.drop_duplicates().reset_index(drop=True)
order_items = order_items.drop_duplicates().reset_index(drop=True)
payments    = payments.drop_duplicates().reset_index(drop=True)
reviews     = reviews.drop_duplicates().reset_index(drop=True)
customers   = customers.drop_duplicates().reset_index(drop=True)
products    = products.drop_duplicates().reset_index(drop=True)
sellers     = sellers.drop_duplicates().reset_index(drop=True)
translation = translation.drop_duplicates().reset_index(drop=True)

dup_reviews = int(reviews.duplicated(subset=["review_id"]).sum())
reviews = reviews.drop_duplicates(subset=["review_id"], keep="first").reset_index(drop=True)
print(f"reviews: dropped {dup_reviews:,} duplicate review_id rows")

geolocation: 1,000,163 -> 720,154 (280,009 dup coords)
reviews: dropped 814 duplicate review_id rows


### Load: Create Empty Schema

In [16]:
engine = create_engine(DATABASE_URL)
metadata = MetaData()

dim_date = Table("dim_date", metadata,
    Column("calendar_date", Date, primary_key=True),
    Column("year", Integer, nullable=False),
    Column("month", Integer, nullable=False),
    Column("day", Integer, nullable=False),
    Column("day_of_week", Integer, nullable=False))

dim_customer = Table("dim_customer", metadata,
    Column("customer_id", String(40), primary_key=True),
    Column("customer_unique_id", String(40), nullable=False),
    Column("customer_zip_code_prefix", String(5), nullable=False),
    Column("customer_city", String(100), nullable=False),
    Column("customer_state", String(2), nullable=False))

dim_seller = Table("dim_seller", metadata,
    Column("seller_id", String(40), primary_key=True),
    Column("seller_zip_code_prefix", String(5), nullable=False),
    Column("seller_city", String(100), nullable=False),
    Column("seller_state", String(2), nullable=False))

dim_product = Table("dim_product", metadata,
    Column("product_id", String(40), primary_key=True),
    Column("product_category_name", String(60)),
    Column("product_category_name_english", String(60)),
    Column("product_name_length", Integer),
    Column("product_description_length", Integer),
    Column("product_photos_qty", Integer),
    Column("product_weight_g", Integer),
    Column("product_length_cm", Integer),
    Column("product_height_cm", Integer),
    Column("product_width_cm", Integer))

dim_geolocation = Table("dim_geolocation", metadata,
    Column("geolocation_zip_code_prefix", String(5), nullable=False),
    Column("geolocation_lat", Float, nullable=False),
    Column("geolocation_lng", Float, nullable=False),
    Column("geolocation_city", String(100), nullable=False),
    Column("geolocation_state", String(2), nullable=False),
    PrimaryKeyConstraint("geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"))

fact_order_items = Table("fact_order_items", metadata,
    Column("order_id", String(40), nullable=False),
    Column("order_item_id", Integer, nullable=False),
    Column("product_id", String(40), ForeignKey("dim_product.product_id"), nullable=False),
    Column("seller_id", String(40), ForeignKey("dim_seller.seller_id"), nullable=False),
    Column("customer_id", String(40), ForeignKey("dim_customer.customer_id"), nullable=False),
    Column("price", Numeric(12, 2), nullable=False),
    Column("freight_value", Numeric(12, 2), nullable=False),
    Column("shipping_limit_date", DateTime, nullable=False),
    PrimaryKeyConstraint("order_id", "order_item_id"))

fact_payments = Table("fact_payments", metadata,
    Column("order_id", String(40), nullable=False),
    Column("payment_sequential", Integer, nullable=False),
    Column("payment_type", String(20), nullable=False),
    Column("payment_installments", Integer, nullable=False),
    Column("payment_value", Numeric(12, 2), nullable=False),
    PrimaryKeyConstraint("order_id", "payment_sequential"))

fact_reviews = Table("fact_reviews", metadata,
    Column("review_id", String(40), primary_key=True),
    Column("order_id", String(40), nullable=False),
    Column("review_score", SmallInteger, nullable=False),
    Column("review_comment_title", Text),
    Column("review_comment_message", Text),
    Column("review_creation_date", DateTime, nullable=False),
    Column("review_answer_timestamp", DateTime, nullable=False))

fact_order_accum = Table("fact_order_accum", metadata,
    Column("order_id", String(40), primary_key=True),
    Column("customer_id", String(40), ForeignKey("dim_customer.customer_id"), nullable=False),
    Column("order_purchase_timestamp", DateTime, nullable=False),
    Column("order_approved_at", DateTime),
    Column("order_delivered_carrier_date", DateTime),
    Column("order_delivered_customer_date", DateTime),
    Column("order_estimated_delivery_date", DateTime, nullable=False),
    Column("current_stage", String(20), nullable=False),
    Column("days_to_deliver", Integer),
    Column("delivery_delay_days", Integer))

fact_daily_snapshot = Table("fact_daily_snapshot", metadata,
    Column("calendar_date", Date, ForeignKey("dim_date.calendar_date"), primary_key=True),
    Column("num_orders", Integer, nullable=False),
    Column("num_orders_delivered", Integer, nullable=False),
    Column("total_order_value", Numeric(14, 2), nullable=False),
    Column("total_freight_value", Numeric(14, 2), nullable=False),
    Column("total_payment_value", Numeric(14, 2), nullable=False))

metadata.drop_all(engine)
metadata.create_all(engine)
print(f"Empty DB ready on {engine.url.drivername} (SQLAlchemy {sa.__version__}).")

Empty DB ready on postgresql+psycopg2 (SQLAlchemy 2.1.4).


In [17]:
def _nullsafe(df):
    """Replace pandas NA (NaN/NaT/<NA>) with None so SQLAlchemy binds NULL."""
    df = df.copy()
    for c in df.columns:
        df[c] = df[c].map(lambda x: None if pd.isna(x) else x)
    return df

dmin = orders["order_purchase_timestamp"].min().normalize()
dmax = orders["order_purchase_timestamp"].max().normalize()
date_spine = pd.DataFrame({"calendar_date": pd.date_range(dmin, dmax, freq="D")})
date_spine["year"]        = date_spine["calendar_date"].dt.year
date_spine["month"]       = date_spine["calendar_date"].dt.month
date_spine["day"]         = date_spine["calendar_date"].dt.day
date_spine["day_of_week"] = date_spine["calendar_date"].dt.dayofweek

dim_customer_df = customers[["customer_id", "customer_unique_id",
                             "customer_zip_code_prefix", "customer_city",
                             "customer_state"]].copy()
dim_seller_df = sellers[["seller_id", "seller_zip_code_prefix",
                         "seller_city", "seller_state"]].copy()
dim_product_df = products.merge(translation, on="product_category_name", how="left")[
    ["product_id", "product_category_name", "product_category_name_english",
     "product_name_length", "product_description_length", "product_photos_qty",
     "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]].copy()
dim_geolocation_df = geolocation[["geolocation_zip_code_prefix", "geolocation_lat",
                                  "geolocation_lng", "geolocation_city",
                                  "geolocation_state"]].copy()

for name, df in [("dim_date", date_spine), ("dim_customer", dim_customer_df),
                 ("dim_seller", dim_seller_df), ("dim_product", dim_product_df),
                 ("dim_geolocation", dim_geolocation_df)]:
    _nullsafe(df).to_sql(name, engine, if_exists="append", index=False)
print("Dimensions loaded (before facts).")

Dimensions loaded (before facts).


### Load Transactional Facts

In [18]:
fact_order_items_df = order_items.merge(
    orders[["order_id", "customer_id"]], on="order_id", how="left")[
    ["order_id", "order_item_id", "product_id", "seller_id", "customer_id",
     "price", "freight_value", "shipping_limit_date"]].copy()
fact_payments_df = payments[["order_id", "payment_sequential", "payment_type",
                             "payment_installments", "payment_value"]].copy()
fact_reviews_df = reviews[["review_id", "order_id", "review_score",
                           "review_comment_title", "review_comment_message",
                           "review_creation_date", "review_answer_timestamp"]].copy()

for name, df in [("fact_order_items", fact_order_items_df),
                 ("fact_payments", fact_payments_df),
                 ("fact_reviews", fact_reviews_df)]:
    _nullsafe(df).to_sql(name, engine, if_exists="append", index=False)
print("Transactional facts loaded.")

Transactional facts loaded.


### Load Accumulating Snapshot

In [19]:
MILESTONES = ["order_approved_at", "order_delivered_carrier_date",
              "order_delivered_customer_date"]

def _dt(v):
    return None if pd.isna(v) else v.to_pydatetime()

UPSERT_ACCUM = text("""
    INSERT INTO fact_order_accum
        (order_id, customer_id, order_purchase_timestamp, order_approved_at,
         order_delivered_carrier_date, order_delivered_customer_date,
         order_estimated_delivery_date, current_stage, days_to_deliver,
         delivery_delay_days)
    VALUES (:order_id, :customer_id, :order_purchase_timestamp, :order_approved_at,
            :order_delivered_carrier_date, :order_delivered_customer_date,
            :order_estimated_delivery_date, :current_stage, :days_to_deliver,
            :delivery_delay_days)
    ON CONFLICT(order_id) DO UPDATE SET
        order_approved_at = COALESCE(excluded.order_approved_at,
                                     fact_order_accum.order_approved_at),
        order_delivered_carrier_date = COALESCE(excluded.order_delivered_carrier_date,
                                                fact_order_accum.order_delivered_carrier_date),
        order_delivered_customer_date = COALESCE(excluded.order_delivered_customer_date,
                                                 fact_order_accum.order_delivered_customer_date),
        current_stage       = excluded.current_stage,
        days_to_deliver     = COALESCE(excluded.days_to_deliver,
                                       fact_order_accum.days_to_deliver),
        delivery_delay_days = COALESCE(excluded.delivery_delay_days,
                                       fact_order_accum.delivery_delay_days)
""")

def load_accumulating(conn, cut_off):
    snap = orders[orders["order_purchase_timestamp"] <= cut_off].copy()
    for c in MILESTONES:
        snap[c] = snap[c].where(snap[c] <= cut_off)
    snap["current_stage"] = np.select(
        [snap["order_delivered_customer_date"].notna(),
         snap["order_delivered_carrier_date"].notna(),
         snap["order_approved_at"].notna()],
        ["delivered", "shipped", "approved"], default="created")
    snap["days_to_deliver"] = (snap["order_delivered_customer_date"]
                               - snap["order_purchase_timestamp"]).dt.days
    snap["delivery_delay_days"] = (snap["order_delivered_customer_date"]
                                   - snap["order_estimated_delivery_date"]).dt.days
    rows = [dict(order_id=r.order_id, customer_id=r.customer_id,
                 order_purchase_timestamp=_dt(r.order_purchase_timestamp),
                 order_approved_at=_dt(r.order_approved_at),
                 order_delivered_carrier_date=_dt(r.order_delivered_carrier_date),
                 order_delivered_customer_date=_dt(r.order_delivered_customer_date),
                 order_estimated_delivery_date=_dt(r.order_estimated_delivery_date),
                 current_stage=r.current_stage,
                 days_to_deliver=None if pd.isna(r.days_to_deliver) else int(r.days_to_deliver),
                 delivery_delay_days=None if pd.isna(r.delivery_delay_days) else int(r.delivery_delay_days))
            for r in snap.itertuples(index=False)]
    conn.execute(UPSERT_ACCUM, rows)
    return len(rows)

CUT1 = orders["order_purchase_timestamp"].quantile(0.5)
CUT2 = orders["order_purchase_timestamp"].max()

def stats(label):
    r = pd.read_sql(text("""SELECT COUNT(*) AS n,
                                   COUNT(order_delivered_customer_date) AS d,
                                   COUNT(DISTINCT order_id) AS u
                            FROM fact_order_accum"""), engine).iloc[0]
    print(f"  {label:12s} rows={r['n']:>7,}  delivered_set={r['d']:>7,}  distinct={r['u']:>7,}")

print("Run 1 (cut-off =", CUT1.strftime("%Y-%m-%d"), ")")
with engine.begin() as conn:
    load_accumulating(conn, CUT1)
stats("after run 1")
pending = pd.read_sql(text("""SELECT order_id FROM fact_order_accum
                              WHERE order_delivered_customer_date IS NULL LIMIT 1"""),
                      engine).iloc[0, 0]
print("  example pending order as of run 1:", pending)

print("Run 2 (cut-off =", CUT2.strftime("%Y-%m-%d"), ")")
with engine.begin() as conn:
    load_accumulating(conn, CUT2)
stats("after run 2")
after = pd.read_sql(text("""SELECT order_id, current_stage, order_delivered_customer_date
                            FROM fact_order_accum WHERE order_id = :oid"""),
                    engine, params={"oid": pending})
print("  same order after run 2 (revisited in place):", after.to_dict("records"))

dup = pd.read_sql(text("SELECT COUNT(*) AS n, COUNT(DISTINCT order_id) AS u FROM fact_order_accum"), engine).iloc[0]
assert dup["n"] == dup["u"], "duplicate order_id in accumulating snapshot"
print("Accumulating snapshot: re-runnable, in-place, no duplicates.")

Run 1 (cut-off = 2018-01-18 )
  after run 1  rows= 49,721  delivered_set= 44,689  distinct= 49,721
  example pending order as of run 1: 136cce7faa42fdb2cefd53fdc79a6098
Run 2 (cut-off = 2018-10-17 )
  after run 2  rows= 99,441  delivered_set= 96,476  distinct= 99,441
  same order after run 2 (revisited in place): [{'order_id': '136cce7faa42fdb2cefd53fdc79a6098', 'current_stage': 'approved', 'order_delivered_customer_date': None}]
Accumulating snapshot: re-runnable, in-place, no duplicates.


### Load: Periodic Snapshot (Dense Daily)

In [20]:
dmin = orders["order_purchase_timestamp"].min().normalize()
dmax = orders["order_purchase_timestamp"].max().normalize()
spine = pd.DataFrame({"calendar_date": pd.date_range(dmin, dmax, freq="D")})

o = orders[["order_id", "order_purchase_timestamp", "order_delivered_customer_date"]].copy()
o["purchase_day"] = o["order_purchase_timestamp"].dt.normalize()
num_orders = o.groupby("purchase_day").size().rename("num_orders")
num_orders_delivered = o[o["order_delivered_customer_date"].notna()] \
                         .groupby("purchase_day").size().rename("num_orders_delivered")

pay = payments.merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id")
pay["day"] = pay["order_purchase_timestamp"].dt.normalize()
total_payment = pay.groupby("day")["payment_value"].sum().rename("total_payment_value")

items = order_items.merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id")
items["day"] = items["order_purchase_timestamp"].dt.normalize()
items["order_value"] = items["price"] + items["freight_value"]
total_order_value = items.groupby("day")["order_value"].sum().rename("total_order_value")
total_freight = items.groupby("day")["freight_value"].sum().rename("total_freight_value")

daily = (spine
         .merge(num_orders,           left_on="calendar_date", right_index=True, how="left")
         .merge(num_orders_delivered, left_on="calendar_date", right_index=True, how="left")
         .merge(total_payment,        left_on="calendar_date", right_index=True, how="left")
         .merge(total_order_value,    left_on="calendar_date", right_index=True, how="left")
         .merge(total_freight,        left_on="calendar_date", right_index=True, how="left")
         .fillna(0))
daily[["num_orders", "num_orders_delivered"]] = \
    daily[["num_orders", "num_orders_delivered"]].astype(int)

zero_days = int((daily["num_orders"] == 0).sum())
print(f"fact_daily_snapshot: {len(daily):,} dense days ({zero_days:,} zero-order days).")
_nullsafe(daily).to_sql("fact_daily_snapshot", engine, if_exists="append", index=False)

fact_daily_snapshot: 774 dense days (140 zero-order days).


774

### Reconcile Row Counts and Monetary Totals by Assertion Statements

In [21]:
def db_scalar(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params).iloc[0, 0]

src_orders    = pd.read_csv(f"{STAGING}/olist_orders_dataset.csv")
src_items     = pd.read_csv(f"{STAGING}/olist_order_items_dataset.csv")
src_payments  = pd.read_csv(f"{STAGING}/olist_order_payments_dataset.csv")
src_reviews   = pd.read_csv(f"{STAGING}/olist_order_reviews_dataset.csv")
src_customers = pd.read_csv(f"{STAGING}/olist_customers_dataset.csv")
src_products  = pd.read_csv(f"{STAGING}/olist_products_dataset.csv")
src_sellers   = pd.read_csv(f"{STAGING}/olist_sellers_dataset.csv")
src_geo       = pd.read_csv(f"{STAGING}/olist_geolocation_dataset.csv",
                            dtype={"geolocation_zip_code_prefix": str})

print("ROW-COUNT RECONCILIATION (db vs source files)")
def check_counts(table, expected, src_name, src_rows, note=""):
    actual = db_scalar(f"SELECT COUNT(*) FROM {table}")
    ok = actual == expected
    print(f"  {'OK ' if ok else 'FAIL'} {table:20s} db={actual:>9,}  "
          f"source({src_name})={src_rows:>9,} {note}")
    assert ok, f"ROW-COUNT MISMATCH in {table}: db={actual} expected={expected}"

check_counts("dim_customer",     len(src_customers), "customers",   len(src_customers))
check_counts("dim_seller",       len(src_sellers),   "sellers",     len(src_sellers))
check_counts("dim_product",      len(src_products),  "products",    len(src_products))
check_counts("fact_order_items", len(src_items),     "order_items", len(src_items))
check_counts("fact_payments",    len(src_payments),  "payments",    len(src_payments))
src_dup_reviews = int(src_reviews.duplicated(subset=["review_id"]).sum())
check_counts("fact_reviews", len(src_reviews) - src_dup_reviews, "reviews", len(src_reviews),
             note=f"(dedup {src_dup_reviews:,} review_id)")
check_counts("fact_order_accum", len(src_orders),    "orders",      len(src_orders))
check_counts("dim_geolocation",  720154,             "geolocation", len(src_geo),
             note="(dedup 1,000,163 -> 720,154)")

snap_orders = db_scalar("SELECT SUM(num_orders) FROM fact_daily_snapshot")
assert snap_orders == len(src_orders), "PERIODIC SNAPSHOT MISMATCH: sum(num_orders) != source orders"
print(f"  {'OK ' if snap_orders == len(src_orders) else 'FAIL'} fact_daily_snapshot   "
      f"sum(num_orders)={snap_orders:>9,}  source(orders)={len(src_orders):>9,}")

print("\nMONETARY RECONCILIATION (db vs source files)")
def money_close(a, b):
    return math.isclose(a, b, rel_tol=1e-6, abs_tol=0.01)

src_pay_sum     = src_payments["payment_value"].sum()
src_price_sum   = src_items["price"].sum()
src_freight_sum = src_items["freight_value"].sum()
checks = [
    ("fact_payments.payment_value",        db_scalar("SELECT SUM(payment_value) FROM fact_payments"), src_pay_sum),
    ("fact_order_items.price",             db_scalar("SELECT SUM(price) FROM fact_order_items"),       src_price_sum),
    ("fact_order_items.freight_value",     db_scalar("SELECT SUM(freight_value) FROM fact_order_items"), src_freight_sum),
    ("daily_snapshot.total_payment_value", db_scalar("SELECT SUM(total_payment_value) FROM fact_daily_snapshot"), src_pay_sum),
    ("daily_snapshot.total_freight_value", db_scalar("SELECT SUM(total_freight_value) FROM fact_daily_snapshot"), src_freight_sum),
    ("daily_snapshot.total_order_value",   db_scalar("SELECT SUM(total_order_value) FROM fact_daily_snapshot"), src_price_sum + src_freight_sum),
]
ok_all = True
for label, dbv, srcv in checks:
    ok = money_close(dbv, srcv)
    ok_all &= ok
    print(f"  {'OK ' if ok else 'FAIL'} {label:42s} db={dbv:>16,.2f}  source={srcv:>16,.2f}")
assert ok_all, "MONETARY RECONCILIATION FAILED"

print("\nETL COMPLETE: all row-count and monetary reconciliations PASSED.")

ROW-COUNT RECONCILIATION (db vs source files)
  OK  dim_customer         db=   99,441  source(customers)=   99,441 
  OK  dim_seller           db=    3,095  source(sellers)=    3,095 
  OK  dim_product          db=   32,951  source(products)=   32,951 
  OK  fact_order_items     db=  112,650  source(order_items)=  112,650 
  OK  fact_payments        db=  103,886  source(payments)=  103,886 
  OK  fact_reviews         db=   98,410  source(reviews)=   99,224 (dedup 814 review_id)
  OK  fact_order_accum     db=   99,441  source(orders)=   99,441 
  OK  dim_geolocation      db=  720,154  source(geolocation)=1,000,163 (dedup 1,000,163 -> 720,154)
  OK  fact_daily_snapshot   sum(num_orders)=   99,441  source(orders)=   99,441

MONETARY RECONCILIATION (db vs source files)
  OK  fact_payments.payment_value                db=   16,008,872.12  source=   16,008,872.12
  OK  fact_order_items.price                     db=   13,591,643.70  source=   13,591,643.70
  OK  fact_order_items.freight_value